# Laboratorio SP8502 - Regresion lineal en Python para Google Colab

Version reconstruida desde cero.  
El notebook usa datos sinteticos y cubre: exploracion, regresion simple, regresion multiple, supuestos, multicolinealidad, transformaciones, observaciones influyentes y comparacion de modelos.

**Regla tecnica aplicada:** las explicaciones estan en celdas Markdown; las celdas de codigo contienen solo Python valido.


In [ ]:
import sys
import subprocess
import importlib.util

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
    "statsmodels": "statsmodels",
    "sklearn": "scikit-learn"
}

missing = []
for import_name, package_name in required_packages.items():
    if importlib.util.find_spec(import_name) is None:
        missing.append(package_name)

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)

print("Environment ready")
print("Python:", sys.version.split()[0])


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.stats import shapiro

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)
np.random.seed(2026)

print("Libraries loaded")


## 1. Crear un conjunto de datos sintetico

El ejemplo simula observaciones de pesca artesanal. La variable dependiente es `captura_kg`. Los predictores incluyen esfuerzo de pesca, experiencia, distancia, temperatura y tipo de embarcacion.


In [ ]:
def generate_fishing_data(n=150, seed=2026):
    rng = np.random.default_rng(seed)

    horas_mar = rng.normal(6.5, 1.8, n).clip(2, 12)
    experiencia_anios = rng.gamma(shape=3.0, scale=3.0, size=n).clip(1, 35)
    distancia_km = rng.normal(18, 7, n).clip(2, 45)
    temperatura_c = rng.normal(27.2, 1.4, n).clip(23, 31)
    embarcacion_motor = rng.binomial(1, 0.62, n)

    error = rng.normal(0, 8, n)
    captura_kg = (
        12
        + 5.2 * horas_mar
        + 0.75 * experiencia_anios
        + 0.90 * distancia_km
        - 1.10 * temperatura_c
        + 11.0 * embarcacion_motor
        + error
    )
    captura_kg = np.maximum(captura_kg, 1)

    df = pd.DataFrame({
        "captura_kg": captura_kg,
        "horas_mar": horas_mar,
        "experiencia_anios": experiencia_anios,
        "distancia_km": distancia_km,
        "temperatura_c": temperatura_c,
        "embarcacion_motor": embarcacion_motor
    })
    df["tipo_embarcacion"] = np.where(df["embarcacion_motor"] == 1, "motor", "sin_motor")
    return df

df = generate_fishing_data()
print(df.shape)
df.head()


In [ ]:
print("Missing values by variable")
print(df.isna().sum())

print("\nDescriptive statistics")
df.describe().T


## 2. Analisis exploratorio


In [ ]:
numeric_cols = ["captura_kg", "horas_mar", "experiencia_anios", "distancia_km", "temperatura_c"]

for col in numeric_cols:
    plt.figure(figsize=(7, 4))
    plt.hist(df[col], bins=20, edgecolor="black")
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()


In [ ]:
corr = df[numeric_cols + ["embarcacion_motor"]].corr()
print(corr.round(3))

plt.figure(figsize=(8, 6))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Pearson correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()


## 3. Regresion lineal simple

Modelo: `captura_kg ~ horas_mar`.


In [ ]:
model_simple = smf.ols("captura_kg ~ horas_mar", data=df).fit()
print(model_simple.summary())


In [ ]:
x = df["horas_mar"]
y = df["captura_kg"]

x_line = np.linspace(x.min(), x.max(), 100)
y_line = model_simple.params["Intercept"] + model_simple.params["horas_mar"] * x_line

plt.figure(figsize=(7, 5))
plt.scatter(x, y, alpha=0.7)
plt.plot(x_line, y_line, linewidth=2)
plt.title("Simple linear regression")
plt.xlabel("Hours at sea")
plt.ylabel("Catch in kg")
plt.show()


## 4. Regresion lineal multiple

Modelo principal: `captura_kg ~ horas_mar + experiencia_anios + distancia_km + temperatura_c + embarcacion_motor`.


In [ ]:
formula_main = "captura_kg ~ horas_mar + experiencia_anios + distancia_km + temperatura_c + embarcacion_motor"
model_main = smf.ols(formula_main, data=df).fit()
print(model_main.summary())


In [ ]:
coef_table = pd.DataFrame({
    "coef": model_main.params,
    "std_error": model_main.bse,
    "t_value": model_main.tvalues,
    "p_value": model_main.pvalues,
    "ci_025": model_main.conf_int()[0],
    "ci_975": model_main.conf_int()[1]
})
coef_table.round(4)


## 5. Diagnostico de supuestos

Se revisan cuatro elementos: linealidad, normalidad aproximada de residuales, homocedasticidad e independencia.


In [ ]:
fitted = model_main.fittedvalues
resid = model_main.resid
std_resid = model_main.get_influence().resid_studentized_internal

plt.figure(figsize=(7, 5))
plt.scatter(fitted, resid, alpha=0.7)
plt.axhline(0, linestyle="--")
plt.title("Residuals vs fitted values")
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.show()

plt.figure(figsize=(7, 5))
sm.qqplot(resid, line="45", fit=True)
plt.title("Q-Q plot of residuals")
plt.show()

plt.figure(figsize=(7, 5))
plt.scatter(fitted, np.sqrt(np.abs(std_resid)), alpha=0.7)
plt.title("Scale-location plot")
plt.xlabel("Fitted values")
plt.ylabel("Sqrt absolute standardized residuals")
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(np.arange(len(resid)), resid, marker="o", linestyle="-", alpha=0.7)
plt.axhline(0, linestyle="--")
plt.title("Residual sequence plot")
plt.xlabel("Observation order")
plt.ylabel("Residuals")
plt.show()


In [ ]:
shapiro_stat, shapiro_p = shapiro(resid)
dw_stat = durbin_watson(resid)

bp_test = het_breuschpagan(resid, model_main.model.exog)
bp_labels = ["LM statistic", "LM p-value", "F statistic", "F p-value"]
bp_results = dict(zip(bp_labels, bp_test))

assumption_tests = pd.DataFrame({
    "test": ["Shapiro-Wilk", "Durbin-Watson", "Breusch-Pagan LM", "Breusch-Pagan F"],
    "statistic": [shapiro_stat, dw_stat, bp_results["LM statistic"], bp_results["F statistic"]],
    "p_value": [shapiro_p, np.nan, bp_results["LM p-value"], bp_results["F p-value"]]
})
assumption_tests.round(4)


## 6. Multicolinealidad mediante VIF

Valores de VIF superiores a 5 sugieren revisar el modelo; superiores a 10 suelen indicar un problema severo de colinealidad.


In [ ]:
X = model_main.model.exog
vif_table = pd.DataFrame({
    "variable": model_main.model.exog_names,
    "vif": [variance_inflation_factor(X, i) for i in range(X.shape[1])]
})
vif_table.round(3)


## 7. Transformacion logaritmica de la variable dependiente

Se ajusta un modelo alternativo con `log(captura_kg)` para comparar ajuste y diagnosticos.


In [ ]:
df["log_captura_kg"] = np.log(df["captura_kg"])

formula_log = "log_captura_kg ~ horas_mar + experiencia_anios + distancia_km + temperatura_c + embarcacion_motor"
model_log = smf.ols(formula_log, data=df).fit()
print(model_log.summary())


In [ ]:
comparison = pd.DataFrame({
    "model": ["main_raw_y", "log_y"],
    "r_squared": [model_main.rsquared, model_log.rsquared],
    "adj_r_squared": [model_main.rsquared_adj, model_log.rsquared_adj],
    "aic": [model_main.aic, model_log.aic],
    "bic": [model_main.bic, model_log.bic],
    "shapiro_p_residuals": [shapiro(model_main.resid)[1], shapiro(model_log.resid)[1]],
    "bp_p_value": [
        het_breuschpagan(model_main.resid, model_main.model.exog)[1],
        het_breuschpagan(model_log.resid, model_log.model.exog)[1]
    ]
})
comparison.round(4)


## 8. Estandarizacion de predictores

La estandarizacion permite comparar magnitudes de predictores medidos en escalas diferentes. Aqui se estandarizan solo las variables X; la variable Y queda en kg.


In [ ]:
scale_cols = ["horas_mar", "experiencia_anios", "distancia_km", "temperatura_c"]
scaler = StandardScaler()

df_std = df.copy()
df_std[[col + "_z" for col in scale_cols]] = scaler.fit_transform(df[scale_cols])

formula_std = "captura_kg ~ horas_mar_z + experiencia_anios_z + distancia_km_z + temperatura_c_z + embarcacion_motor"
model_std = smf.ols(formula_std, data=df_std).fit()
print(model_std.summary())


## 9. Observaciones influyentes

Se calculan leverage, distancia de Cook y residuales studentizados.


In [ ]:
influence = model_main.get_influence()
influence_df = pd.DataFrame({
    "obs_id": np.arange(len(df)),
    "fitted": model_main.fittedvalues,
    "residual": model_main.resid,
    "studentized_residual": influence.resid_studentized_internal,
    "leverage": influence.hat_matrix_diag,
    "cooks_d": influence.cooks_distance[0]
})

n = len(df)
p = int(model_main.df_model) + 1
cook_threshold = 4 / n
leverage_threshold = 2 * p / n

influence_df["high_cook"] = influence_df["cooks_d"] > cook_threshold
influence_df["high_leverage"] = influence_df["leverage"] > leverage_threshold
influence_df.sort_values("cooks_d", ascending=False).head(10).round(4)


In [ ]:
plt.figure(figsize=(7, 5))
plt.stem(influence_df["obs_id"], influence_df["cooks_d"], basefmt=" ")
plt.axhline(cook_threshold, linestyle="--")
plt.title("Cook distance by observation")
plt.xlabel("Observation id")
plt.ylabel("Cook distance")
plt.show()

plt.figure(figsize=(7, 5))
plt.scatter(influence_df["leverage"], influence_df["studentized_residual"], alpha=0.7)
plt.axvline(leverage_threshold, linestyle="--")
plt.axhline(2, linestyle="--")
plt.axhline(-2, linestyle="--")
plt.title("Leverage vs studentized residuals")
plt.xlabel("Leverage")
plt.ylabel("Studentized residuals")
plt.show()


## 10. Comparacion de modelos candidatos


In [ ]:
models = {
    "m1_simple_hours": smf.ols("captura_kg ~ horas_mar", data=df).fit(),
    "m2_effort_experience": smf.ols("captura_kg ~ horas_mar + experiencia_anios", data=df).fit(),
    "m3_core": smf.ols("captura_kg ~ horas_mar + experiencia_anios + distancia_km", data=df).fit(),
    "m4_full": model_main,
    "m5_log_y": model_log
}

model_compare = []
for name, mod in models.items():
    model_compare.append({
        "model": name,
        "n_params": int(mod.df_model) + 1,
        "r_squared": mod.rsquared,
        "adj_r_squared": mod.rsquared_adj,
        "aic": mod.aic,
        "bic": mod.bic
    })

model_compare = pd.DataFrame(model_compare).sort_values("aic")
model_compare.round(4)


## 11. Funcion reutilizable para diagnostico rapido

Esta funcion permite aplicar el mismo diagnostico a cualquier modelo ajustado con `statsmodels`.


In [ ]:
def regression_diagnostics(model):
    resid = model.resid
    exog = model.model.exog
    infl = model.get_influence()
    n_obs = int(model.nobs)
    n_params = int(model.df_model) + 1

    shapiro_stat, shapiro_p = shapiro(resid)
    bp = het_breuschpagan(resid, exog)
    dw = durbin_watson(resid)

    result = {
        "n_obs": n_obs,
        "n_params": n_params,
        "r_squared": model.rsquared,
        "adj_r_squared": model.rsquared_adj,
        "aic": model.aic,
        "bic": model.bic,
        "shapiro_p": shapiro_p,
        "breusch_pagan_p": bp[1],
        "durbin_watson": dw,
        "max_cooks_d": float(np.max(infl.cooks_distance[0])),
        "cook_threshold_4_over_n": 4 / n_obs,
        "max_leverage": float(np.max(infl.hat_matrix_diag)),
        "leverage_threshold_2p_over_n": 2 * n_params / n_obs
    }
    return pd.Series(result)

regression_diagnostics(model_main).round(4)


## 12. Exportar resultados opcionales

Esta celda guarda las tablas principales en archivos CSV dentro del entorno de Colab.


In [ ]:
coef_table.to_csv("coeficientes_modelo_principal.csv")
vif_table.to_csv("vif_modelo_principal.csv", index=False)
influence_df.to_csv("diagnostico_influencia.csv", index=False)
model_compare.to_csv("comparacion_modelos.csv", index=False)

print("CSV files created in the current runtime")
